In [ ]:
import pandas as pd
import numpy as np

Define Variables

In [ ]:
INITIAL_CASH = 100000
LOT_SIZE = 100
DATA_PATH = "interday (intel-assignment).csv"
ANNUAL_FEE_RATE = 0.0005  # 0.05%
DAILY_FEE_RATE = ANNUAL_FEE_RATE / 365
start_date = pd.to_datetime("2018-12-31")
end_date = pd.to_datetime("2021-12-31")
MAX_HOLDING_STOCKS_NUMBER = 4

In [ ]:
rank_schedule = {
    "2018-12-31": [2573042, 2572286, 1232815, 2572066],
    "2019-03-31": [2573125, 3695, 2572066, 2572067],
    "2019-06-30": [3695, 8893, 2572065, 2572067],
    "2019-09-30": [2573062, 2572286, 3695, 2572067],
    "2019-12-31": [2572856, 2572065, 851607, 2572765],
    "2020-03-31": [2573062, 851607, 497280, 2572066],
    "2020-06-30": [851607, 2572066, 497280, 2572067],
    "2020-09-30": [4572, 851607, 2572066, 1232815],
    "2020-12-31": [4572, 2572067, 2572765, 3695],
    "2021-03-31": [1232815, 2572067, 2572065, 7634],
    "2021-06-30": [2573085, 2572065, 2573062, 497280],
    "2021-09-30": [7634, 497280, 3695, 1232815],
}


Create Intraday Stock Price Dataframe

In [ ]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
prices = df.pivot_table(index="timestamp", columns="jitta_stock_id", values="close")

In [ ]:
prices

In [ ]:
rank_ids = sorted({sid for stocks in rank_schedule.values() for sid in stocks})

In [ ]:
rank_ids = set(rank_ids)  # จากเซลล์ก่อนหน้า
excluded_ids = sorted(set(prices.columns) - rank_ids)
excluded_ids[:20], len(excluded_ids)

In [ ]:
tx_rows = []

def log_tx(date, action, stock_id, shares, price, amount, cash_after, note):
    tx_rows.append({
        "date": date,
        "action": action,
        "stock_id": stock_id,
        "shares": shares,
        "price": price,
        "amount": amount,
        "cash_after": cash_after,
        "note": note,
    })


In [ ]:
rank_ids = sorted({sid for stocks in rank_schedule.values() for sid in stocks})
df_prices = prices.reindex(columns=rank_ids)
trading_dates = df_prices.index 
def map_to_trading_date(raw_date):
    if raw_date in trading_dates:
        return raw_date
    pos = trading_dates.searchsorted(raw_date)
    if pos >= len(trading_dates):
        raise ValueError(f"No trading date on or after {raw_date}")
    return trading_dates[pos]
effective_schedule = {
    map_to_trading_date(pd.to_datetime(d)): stocks
    for d, stocks in rank_schedule.items()
}

In [ ]:
effective_schedule

In [ ]:
def rebalance(date, target_stocks, holdings, cash, lot_size=LOT_SIZE):
    price_row = daily_prices.loc[date]  
    total_value = cash
    for sid, shares in holdings.items():
        if shares == 0:
            continue
        total_value += shares * price_row[sid]

    target_value = total_value / len(target_stocks)

    desired = {}
    for sid in target_stocks:
        px = price_row[sid]
        lots = np.floor(target_value / (px * lot_size))
        desired[sid] = int(lots * lot_size)

    # SELL first
    all_sids = sorted(set(holdings.keys()) | set(target_stocks))
    for sid in all_sids:
        cur = holdings.get(sid, 0)
        tgt = desired.get(sid, 0)
        if cur > tgt:
            shares = cur - tgt
            proceeds = shares * price_row[sid]
            cash += proceeds
            holdings[sid] = tgt
            log_tx(date, "SELL", sid, shares, price_row[sid], proceeds, cash, "rebalance")

    # BUY after
    for sid in target_stocks:
        cur = holdings.get(sid, 0)
        tgt = desired.get(sid, 0)
        if cur < tgt:
            shares = tgt - cur
            cost = shares * price_row[sid]
            if cost > cash + 1e-9:
                max_lots = int(cash // (price_row[sid] * lot_size))
                shares = max_lots * lot_size
                if shares == 0:
                    continue
                cost = shares * price_row[sid]
                tgt = cur + shares
            cash -= cost
            holdings[sid] = tgt
            log_tx(date, "BUY", sid, shares, price_row[sid], -cost, cash, "rebalance")

    return holdings, cash


In [ ]:
stock_cols = [f"stock{i}" for i in range(1, MAX_HOLDING_STOCKS_NUMBER + 1)]
lot_cols = [f"lot{i}" for i in range(1, MAX_HOLDING_STOCKS_NUMBER + 1)]
value_cols = [f"value{i}" for i in range(1, MAX_HOLDING_STOCKS_NUMBER + 1)]

holdings = {}
cash = INITIAL_CASH
accrued_fee = 0.0
current_order = []

nav_rows = []
rank_schedule_dt = {pd.to_datetime(k): v for k, v in rank_schedule.items()}
rebalance_dates = pd.to_datetime(list(rank_schedule.keys()))
rebalance_schedule = set(rebalance_dates)


# print(rebalance_schedule)
dates = pd.date_range(start_date, end_date, freq="D")
daily_prices = df_prices.reindex(dates).ffill()
for date in dates:
    is_rebalance = False
    is_payfee = False

    if date in effective_schedule:
        current_order = effective_schedule[date]
        holdings, cash = rebalance(date, current_order, holdings, cash)
        is_rebalance = True
       
    held = [sid for sid in current_order if holdings.get(sid, 0) > 0]
    held = held[:MAX_HOLDING_STOCKS_NUMBER]
    held = held + [None] * (MAX_HOLDING_STOCKS_NUMBER - len(held))

    price_row = daily_prices.loc[date]
    stock_value_total = 0.0
    for sid, shares in holdings.items():
        if shares == 0:
            continue
        stock_value_total += shares * price_row[sid]

    asset_value = stock_value_total + cash - accrued_fee
    daily_fee = asset_value * DAILY_FEE_RATE
    accrued_fee += daily_fee

    if date.month == 12 and date.day == 31 and accrued_fee > 0:
        cash -= accrued_fee
        log_tx(date, "FEE_PAYMENT", None, None, None, -accrued_fee, cash, "year-end")
        accrued_fee = 0.0
        is_payfee = True
    lots = []
    values = []
    for sid in held:
        if sid is None:
            lots.append(0)
            values.append(0.0)
        else:
            shares = holdings[sid]
            lots.append(shares // LOT_SIZE)
            values.append(shares * price_row[sid])
    nav = stock_value_total + cash - accrued_fee
    nav_rows.append({
        "date": date,
        **{stock_cols[i]: held[i] for i in range(MAX_HOLDING_STOCKS_NUMBER)},
        **{lot_cols[i]: lots[i] for i in range(MAX_HOLDING_STOCKS_NUMBER)},
        **{value_cols[i]: values[i] for i in range(MAX_HOLDING_STOCKS_NUMBER)},
        "stock_value_total": sum(values),
        "cash": cash,
        "accrued_fee": accrued_fee,
        "nav": nav,
        "is_rebalance": is_rebalance,
        "is_payfee": is_payfee,
    })

df_nav = pd.DataFrame(nav_rows).set_index("date").sort_index()
df_trade_log = pd.DataFrame(tx_rows).sort_values("date").reset_index(drop=True)
df_trade_log.head()


In [ ]:
df_nav.to_csv("nav.csv", index=True)
df_trade_log.to_csv("trade_log.csv", index=True)

In [ ]:
df_trade_log

In [ ]:
def calc_return_metrics(df_nav):
    nav_start = df_nav["nav"].iloc[0]
    nav_end = df_nav["nav"].iloc[-1]
    years = (df_nav.index[-1] - df_nav.index[0]).days / 365

    total_return = (nav_end - nav_start) / nav_start
    cagr = (nav_end / nav_start) ** (1 / years) - 1

    return {
        "NAV_start": nav_start,
        "NAV_end": nav_end,
        "Total_Return": total_return,
        "CAGR": cagr,
        "Years": years,
    }

In [ ]:
return_metrix = calc_return_metrics(df_nav)
print(return_metrix)